In [ ]:
import requests
import xarray as xr
import numpy as np
import scipy



In [9]:
def download_and_open(url,sfc = True, atm = True, filename="temp.grib2"):

  
  print("Downloading filtered data...")
  response = requests.get(url, stream=True)
  response.raise_for_status()

  with open(filename, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
      f.write(chunk)

      # 3. Open with xarray using the cfgrib engine
  if sfc:
    ds_sfc = xr.open_dataset(filename, 
                       filter_by_keys={'typeOfLevel': 'meanSea' },
                       engine='cfgrib')
  else:
    ds_sfc = None
  if atm:
    ds_atm = xr.open_dataset(filename, 
                       filter_by_keys={'typeOfLevel': 'isobaricInhPa' },
                       engine='cfgrib')
  else:
    ds_atm = None
  
  return ds_sfc, ds_atm  

In [10]:
url = 'https://noaa-nws-hafs-pds.s3.amazonaws.com/hfsa/20230927/12/17l.2023092712.hfsa.storm.atm.f024.grb2'

atm_args = dict(typeOfKey = 'isobaricInhPa',filename="Model_Data/temp_gribs/atm_temp.grib2" )
sfc_args = dict(typeOfKey = 'meanSea',filename="Model_Data/temp_gribs/sfc_temp.grb2" )


ds_sfc, ds_atm = download_and_open(url)

Ignoring index file 'temp.grib2.5b7b6.idx' older than GRIB file
Ignoring index file 'temp.grib2.5b7b6.idx' older than GRIB file
skipping variable: paramId==3017 shortName='dpt'
Traceback (most recent call last):
  File "c:\Users\12ian\anaconda3\envs\ATMS523-Proj-Everything-Else\Lib\site-packages\cfgrib\dataset.py", line 726, in build_dataset_components
    dict_merge(variables, coord_vars)
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\12ian\anaconda3\envs\ATMS523-Proj-Everything-Else\Lib\site-packages\cfgrib\dataset.py", line 642, in dict_merge
    raise DatasetBuildError(
    ...<2 lines>...
    )
cfgrib.dataset.DatasetBuildError: key present and new value is different: key='isobaricInhPa' value=Variable(dimensions=('isobaricInhPa',), data=array([1000.,  975.,  950.,  925.,  900.,  875.,  850.,  825.,  800.,
        775.,  750.,  725.,  700.,  675.,  650.,  625.,  600.,  575.,
        550.,  525.,  500.,  475.,  450.,  425.,  400.,  375.,  350.,
        325.,  300.,  275.,  2

In [ ]:
xr.open_dataset('temp.grib2')

In [ ]:
def get_sfc_center(ds):
    min_pressure = ds['prmsl'].min().values
    if min_pressure > 100500: # Pressure threshold (pa) 1005mb
        return None
    nan_mask = ds['prmsl'].isnull()
    ds_sfc = ds.where(~nan_mask, drop = True).isel(latitude = slice(50,-50), longitude = slice(50,-50))
    

    slp_threshold = np.quantile(ds_sfc['prmsl'], 0.01)
    slp_clusters = ds_sfc['prmsl'].where(ds_sfc['prmsl'] < slp_threshold)
    slp_clusters_mask = ~slp_clusters.isnull()

    clusters = scipy.ndimage.label(slp_clusters_mask)
    cluster_size = []
    for a in np.arange(clusters[1]):
        cluster_number = a + 1
        cluster_size.append(np.sum(np.where(clusters[0] == cluster_number, 1, 0)))

    
    largest_cluster = np.argmax(cluster_size) + 1
    center_y, center_x = scipy.ndimage.center_of_mass(clusters[0], largest_cluster)
    mslp = ds_sfc['prmsl'].where(clusters[0] == largest_cluster).min().values

    center_lat = ds_sfc.latitude.isel(latitude =int(np.round(center_y)))
    center_lon = ds_sfc.longitude.isel(longitude =int(np.round(center_x)))
    
    center_info = dict(center_lat = center_lat, center_lon = center_lon, mslp = mslp)
    return center_info


print(get_sfc_center(ds_sfc))

In [ ]:
ds_sfc['prmsl'].plot()